In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd
pd.set_option('display.max_columns', None)

In [ ]:
#!/usr/bin/env python3
"""
harvest_authorships.py
======================
Recover per-author country linkage for the tick / TBD corpus.

WHY THIS EXISTS
---------------
The flattened `llm_screen_test.csv` set-deduplicated `oa_countries`
(e.g. an 11-author paper collapses to "FR | US | BR | DZ | AU"), so the
mapping author_position -> institution -> country is GONE. Paper 3's core
metrics (local first-author share, local last-author share, parachute index)
are therefore not computable from the CSV as it stands.

This script re-fetches ONLY the `authorships` array from OpenAlex, keyed on the
`oa_id` you already have, and reshapes it into position-aligned, pipe-delimited
vectors that line up 1:1 with the existing `oa_author_orcids` column (same author
order, blanks preserved). Re-fetched JSON is clean UTF-8, so the mojibake in the
old columns does not propagate here (use clean_existing_columns() for the columns
you keep from the old file).

OUTPUT (one row per oa_id) — author-aligned vectors use '|' BETWEEN authors and
';' WITHIN an author (an author can sit in >1 institution/country):
    oa_id
    au_n                       int   (authors in authorships[])
    au_position_vec            first|middle|...|last
    au_id_vec                  A5001|A5002|...          (OpenAlex author IDs)
    au_orcid_vec               0000-..| |0000-..        (canonical, re-fetched)
    au_country_vec             ZA;FR| |US               (each author's inst countries)
    au_ror_vec                 053cj9d07;05q3vnk25| |...
    au_insttype_vec            education;government| |...
    au_iscorr_vec              True|False|...
    n_authors_no_affil         int   (authors OpenAlex could not geolocate)
    first_author_countries     ZA;FR (set for the 'first' author)
    last_author_countries      FR
    corresponding_countries    FR;US
    all_author_countries       ZA|FR|US|...             (ordered multiset, NOT deduped)

DEPENDENCIES
    pip install requests pandas ftfy
    (pyarrow optional, only if you switch OUT_PATH to .parquet)

USAGE
    python harvest_authorships.py \
        --in  llm_screen_test.csv \
        --out authorships_reharvested.csv \
        --mailto cognosxx@gmail.com            # polite pool -> faster, more reliable
        # --api-key YOUR_KEY           # optional, if you have one

The script is RESUMABLE: rerun the same command after any interruption and it
skips oa_ids already present in --out.
"""

import argparse
import json
import sys
import time
from pathlib import Path
import pandas as pd, requests

OPENALEX = "https://api.openalex.org/works"
BATCH = 50          # OR up to 50 ids per request (safe URL length)
SELECT = "id,authorships"
MAX_RETRIES = 5
BASE_BACKOFF = 2.0  # seconds; doubles each retry


# --------------------------------------------------------------------------- #
# 1. ID loading                                                               #
# --------------------------------------------------------------------------- #
def short_id(x: str) -> str:
    """'https://openalex.org/W123' or 'W123' -> 'W123'."""
    if not isinstance(x, str):
        return ""
    return x.rsplit("/", 1)[-1].strip()


def load_oa_ids(csv_path: str) -> list[str]:
    """Read canonical oa_ids only, deduplicated, preserving order."""
    # usecols keeps memory low on the 396 MB file
    df = pd.read_csv(
        csv_path,
        usecols=["oa_id", "is_canonical"],
        dtype=str,
        on_bad_lines="skip",
    )
    df = df[df["is_canonical"].str.upper() == "TRUE"]
    ids = [short_id(x) for x in df["oa_id"].dropna().unique() if short_id(x)]
    print(f"[load] {len(ids):,} canonical oa_ids to fetch", file=sys.stderr)
    return ids


def already_done(out_path: str) -> set[str]:
    p = Path(out_path)
    if not p.exists():
        return set()
    done = set(pd.read_csv(p, usecols=["oa_id"], dtype=str)["oa_id"])
    print(f"[resume] {len(done):,} already fetched, skipping", file=sys.stderr)
    return done


# --------------------------------------------------------------------------- #
# 2. Fetching (batched, retried, rate-limit aware)                            #
# --------------------------------------------------------------------------- #
def fetch_batch(ids: list[str], session: requests.Session,
                mailto: str | None, api_key: str | None) -> list[dict]:
    params = {
        "filter": f"ids.openalex:{'|'.join(ids)}",
        "select": SELECT,
        "per-page": str(len(ids)),
    }
    if mailto:
        params["mailto"] = mailto
    if api_key:
        params["api_key"] = api_key

    for attempt in range(MAX_RETRIES):
        try:
            r = session.get(OPENALEX, params=params, timeout=60)
            if r.status_code == 200:
                return r.json().get("results", [])
            if r.status_code in (429, 500, 502, 503):
                wait = BASE_BACKOFF * (2 ** attempt)
                print(f"[retry] HTTP {r.status_code}, sleeping {wait:.0f}s",
                      file=sys.stderr)
                time.sleep(wait)
                continue
            r.raise_for_status()
        except requests.RequestException as e:
            wait = BASE_BACKOFF * (2 ** attempt)
            print(f"[retry] {e}; sleeping {wait:.0f}s", file=sys.stderr)
            time.sleep(wait)
    print(f"[error] batch failed after {MAX_RETRIES} retries: {ids[:3]}...",
          file=sys.stderr)
    return []


# --------------------------------------------------------------------------- #
# 3. Reshape one work -> aligned vectors                                      #
# --------------------------------------------------------------------------- #
def parse_authorships(work: dict) -> dict:
    oa_id = short_id(work.get("id", ""))
    auths = work.get("authorships", []) or []

    positions, ids_, orcids, corr = [], [], [], []
    country_per_author, ror_per_author, type_per_author = [], [], []
    first_c, last_c, corr_c, all_c = set(), set(), set(), []
    n_no_affil = 0

    for a in auths:
        pos = a.get("author_position", "") or ""
        author = a.get("author", {}) or {}
        aid = short_id(author.get("id", ""))
        orcid = short_id(author.get("orcid", "")) if author.get("orcid") else ""
        is_corr = bool(a.get("is_corresponding", False))

        insts = a.get("institutions", []) or []
        c_set, ror_set, type_set = [], [], []
        for inst in insts:
            cc = inst.get("country_code") or ""
            if cc:
                c_set.append(cc)
            if inst.get("ror"):
                ror_set.append(short_id(inst["ror"]))
            if inst.get("type"):
                type_set.append(inst["type"])
        # fallback: newer API exposes deduped authorships[].countries
        if not c_set and a.get("countries"):
            c_set = list(a.get("countries")) # FIX: Corrected syntax to get 'countries' list
        if not c_set:
            n_no_affil += 1

        positions.append(pos)
        ids_.append(aid)
        orcids.append(orcid)
        corr.append(str(is_corr))
        country_per_author.append(";".join(dict.fromkeys(c_set)))   # keep order, dedup
        ror_per_author.append(";".join(dict.fromkeys(ror_set)))
        type_per_author.append(";".join(dict.fromkeys(type_set)))

        all_c.extend(c_set)
        if pos == "first":
            first_c.update(c_set)
        if pos == "last":
            last_c.update(c_set)
        if is_corr:
            corr_c.update(c_set)

    # single-author papers: OpenAlex marks the sole author 'first'; treat as last too
    if len(auths) == 1 and not last_c:
        last_c = set(first_c)

    return {
        "oa_id": oa_id,
        "au_n": len(auths),
        "au_position_vec": "|".join(positions),
        "au_id_vec": "|".join(ids_),
        "au_orcid_vec": "|".join(orcids),
        "au_country_vec": "|".join(country_per_author),
        "au_ror_vec": "|".join(ror_per_author),
        "au_insttype_vec": "|".join(type_per_author),
        "au_iscorr_vec": "|".join(corr),
        "n_authors_no_affil": n_no_affil,
        "first_author_countries": ";".join(sorted(first_c)),
        "last_author_countries": ";".join(sorted(last_c)),
        "corresponding_countries": ";".join(sorted(corr_c)),
        "all_author_countries": "|".join(all_c),
    }


# --------------------------------------------------------------------------- #
# 4. Main loop                                                                #
# --------------------------------------------------------------------------- #
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--in", dest="inp", required=True, help="llm_screen_test.csv")
    ap.add_argument("--out", dest="out", required=True)
    ap.add_argument("--mailto", default=None, help="email for OpenAlex polite pool")
    ap.add_argument("--api-key", dest="api_key", default=None)
    args = ap.parse_args()

    ids  = load_oa_ids(args.inp)
    done = already_done(args.out)
    todo = [i for i in ids if i not in done]
    print(f"[plan] {len(todo):,} remaining", file=sys.stderr)

    session = requests.Session()
    session.headers.update({"User-Agent": "tick-sos-study/1.0"})
    write_header = not Path(args.out).exists()
    buf = []

    for start in range(0, len(todo), BATCH):
        batch = todo[start:start + BATCH]
        works = fetch_batch(batch, session, args.mailto, args.api_key)
        returned = {short_id(w.get("id","" )) for w in works}
        for mid in set(batch) - returned:
            buf.append({"oa_id": mid, "au_n": 0, "au_position_vec": "", "au_id_vec": "",
                        "au_orcid_vec": "", "au_country_vec": "", "au_ror_vec": "",
                        "au_insttype_vec": "", "au_iscorr_vec": "", "n_authors_no_affil": 0,
                        "first_author_countries": "", "last_author_countries": "",
                        "corresponding_countries": "", "all_author_countries": ""})
        for w in works:
            buf.append(parse_authorships(w))
        if len(buf) >= 2000:
            pd.DataFrame(buf).to_csv(args.out, mode="a", header=write_header, index=False)
            write_header = False; buf.clear()
            print(f"[progress] {start+len(batch):,}/{len(todo):,}", file=sys.stderr)
        time.sleep(0.12)

    if buf:
        pd.DataFrame(buf).to_csv(args.out, mode="a", header=write_header, index=False)
    print(f"[done] wrote {args.out}", file=sys.stderr)


# --------------------------------------------------------------------------- #
# 5. UTILITIES you run separately                                             #
# --------------------------------------------------------------------------- #
def clean_existing_columns(in_csv: str, out_csv: str):
    """
    Fix mojibake ('AgustÃ­n Estradaâ€PeÃ±a' -> 'Agustín Estrada-Peña') in the
    text columns you KEEP from the old CSV. Run once; the re-harvested
    authorship columns above are already clean and don't need this.

        python -c "import harvest_authorships as h; \
                   h.clean_existing_columns('llm_screen_test.csv','clean.csv')"
    """
    import ftfy
    text_cols = ["oa_title", "oa_authors", "oa_journal", "oa_abstract",
                 "oa_keywords", "oa_affiliations_raw", "oa_institutions",
                 "gpt_reason", "gemini_reason", "deepseek_reason"]
    df = pd.read_csv(in_csv, dtype=str, on_bad_lines="skip")
    for c in text_cols:
        if c in df.columns:
            df[c] = df[c].map(lambda x: ftfy.fix_text(x) if isinstance(x, str) else x)
    df.to_csv(out_csv, index=False)
    print(f"[ftfy] cleaned {len(df):,} rows -> {out_csv}", file=sys.stderr)


def compute_locality(row: pd.Series, target_iso: set[str]) -> dict:
    """
    TEMPLATE for Paper 3 once you have the study-location bloc for a paper.
    `target_iso` = ISO-2 codes for the bloc the paper is 'about'
    (e.g. SSA set, or Maghreb = {'MA','DZ','TN','LY','EG','MR'}).

    Returns the three headline equity flags. An author is 'local' if ANY of
    their institution countries is in target_iso.
    """
    def to_set(s):
        return set(str(s).split(";")) - {""} if pd.notna(s) else set()

    first = to_set(row.get("first_author_countries"))
    last = to_set(row.get("last_author_countries"))
    any_local = target_iso & to_set(row.get("all_author_countries").replace("|", ";"))

    return {
        "local_first_author": bool(first & target_iso),
        "local_last_author": bool(last & target_iso),
        "parachute": len(any_local) == 0,   # no local author anywhere on the paper
    }



In [ ]:
from google.colab import userdata
OPENALEX_API_KEY = userdata.get('OPENALEX_API_KEY')

In [ ]:
import sys

sys.argv = [
    "harvest",
    "--in",  "/content/drive/MyDrive/COGNOSX/ticks/preprocessing_data/llm_screen/llm_screen_test.csv",
    "--out", "/content/drive/MyDrive/COGNOSX/ticks/preprocessing_data/authorships_reharvested.csv",
    "--mailto", "cognosxx@gmail.com",
    "--api-key", OPENALEX_API_KEY,

]

main()